# ML-05 — Feature Vector and Leakage / Privacy Check

This audit creates a public-safe starter feature vector and documents what is excluded before model fitting.

## 1. Build the feature vector

The vector uses non-identifying observed search, engagement, freshness, age, and content-metadata signals. Missingness is represented with flags before numerical filling, because blanket `fillna(0)` can encode content type.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

base = ['impressions_90d','clicks_90d','sessions_90d','avg_position','content_age_days','days_since_update','ctr','engagement_rate','scroll_rate']
base = [c for c in base if c in df]
X = df[base].copy()
for col in base:
    if X[col].isna().any():
        X[f'has_{col}'] = X[col].notna().astype('int8')
        X[col] = X[col].fillna(X[col].median())
print('feature rows, columns=', X.shape)

feature rows, columns= (30000, 9)


## 2. Label boundary

`is_declining_label` is the teaching-slice target proxy. It comes from `trend_direction`, which comes from `trend_pct`; none of those fields can enter `X`. For a production claim, a future-window outcome and time-forward validation would be required.

In [2]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

forbidden = {'content_id','client_id','trend_direction','trend_pct','is_declining_label','impressions_last_30d','impressions_prev_30d'}
print('forbidden fields present in X=', sorted(forbidden.intersection(X.columns)))
assert not forbidden.intersection(X.columns)

forbidden fields present in X= []


## 3. Privacy and identifier check

Client/content identifiers are retained only as split or join keys outside the feature matrix. Client names, domains, URLs, titles, keywords, raw queries, and credentials are not present in this public notebook.

In [3]:
from pathlib import Path
import pandas as pd

DATA = Path('data/raw/content_refresh_anonymized.csv')
if not DATA.exists():
    DATA = Path.cwd().parents[1] / 'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA)
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = df['trend_direction'].eq('down').astype('int8')

identifier_columns = [c for c in ['content_id','client_id'] if c in df]
print('identifier columns kept outside X:', identifier_columns)
print('X contains identifiers:', any(c in X.columns for c in identifier_columns))

identifier columns kept outside X: ['content_id', 'client_id']
X contains identifiers: False


## 4. Verdict

**Pass for the starter-slice experiment:** known leakage fields and identifiers are excluded from the feature matrix, while missingness is handled explicitly. **Limit:** features and label are still from a snapshot, so the capstone reports a grouped-split ranking comparison rather than future causal recovery prediction.

## Self-check

- [x] Feature vector built in code.
- [x] Label-derived and identifier fields excluded.
- [x] No private data displayed.
- [x] Honest temporal limitation stated.

In [4]:
print('Leakage verdict: PASS for excluded fields; temporal limitation retained.')

Leakage verdict: PASS for excluded fields; temporal limitation retained.
